# ClusterGuard 분석 플로우 시각화

DeepAgent + LangGraph 통합 흐름 (master_logs 주입 포함)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(18, 22))
ax.set_xlim(0, 18)
ax.set_ylim(0, 22)
ax.axis('off')
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

# ── 색상 팔레트 ──────────────────────────────────────────────
C_AGENT   = '#1f6feb'   # DeepAgent 단계 — 파랑
C_LG      = '#388bfd'   # LangGraph 내부 — 밝은 파랑
C_SSH     = '#f78166'   # SSH / NodeLogFetcher — 붉은 계열
C_CH      = '#56d364'   # ClickHouse — 초록
C_SYNTH   = '#d2a8ff'   # synthesize — 보라
C_ARROW   = '#8b949e'   # 기본 화살표
C_NEW     = '#ffa657'   # 신규 추가 요소 — 주황
C_TEXT    = '#e6edf3'
C_SUB     = '#8b949e'

def box(ax, x, y, w, h, label, sublabel='', color=C_AGENT, alpha=0.85, fontsize=10):
    rect = FancyBboxPatch((x, y), w, h,
                          boxstyle='round,pad=0.05',
                          facecolor=color, edgecolor='#30363d',
                          linewidth=1.5, alpha=alpha, zorder=3)
    ax.add_patch(rect)
    cy = y + h / 2
    if sublabel:
        ax.text(x + w/2, cy + 0.12, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color='#0d1117', zorder=4)
        ax.text(x + w/2, cy - 0.22, sublabel, ha='center', va='center',
                fontsize=7.5, color='#0d1117', alpha=0.8, zorder=4)
    else:
        ax.text(x + w/2, cy, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color='#0d1117', zorder=4)

def arrow(ax, x1, y1, x2, y2, color=C_ARROW, label='', lw=1.5):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw),
                zorder=2)
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.08, my, label, fontsize=7.5, color=color, zorder=5)

def section_bg(ax, x, y, w, h, label, color):
    rect = FancyBboxPatch((x, y), w, h,
                          boxstyle='round,pad=0.1',
                          facecolor=color, edgecolor=color,
                          linewidth=1, alpha=0.08, zorder=1)
    ax.add_patch(rect)
    ax.text(x + 0.15, y + h - 0.25, label, fontsize=8, color=color,
            alpha=0.7, fontweight='bold', zorder=2)

# ════════════════════════════════════════════════════════════
# 제목
# ════════════════════════════════════════════════════════════
ax.text(9, 21.4, 'ClusterGuard 분석 플로우', ha='center', va='center',
        fontsize=16, fontweight='bold', color=C_TEXT)
ax.text(9, 21.0, 'DeepAgent  +  LangGraph  +  master_logs SSH 주입',
        ha='center', va='center', fontsize=10, color=C_SUB)

# ════════════════════════════════════════════════════════════
# DEEPAGENT 단계 (왼쪽 열, x=0.4)
# ════════════════════════════════════════════════════════════
section_bg(ax, 0.2, 0.2, 6.8, 20.3, 'DeepAgent (ReAct loop)', C_AGENT)

stages = [
    (20.0, '트리거 수신', 'slowlog_timestamp / kafka_receive_time'),
    (18.5, '1단계: 초기 파악', 'cluster_health  +  check_new_slowlogs'),
    (17.0, '2단계: 유입 안정화', 'sleep(30) × N  →  zero_streak == 2'),
    (15.5, '3단계: 구간 결정', 'first_seen−2m  ~  last_seen+1m'),
    (14.0, '4단계: analyze_logs', 'start_iso / end_iso  →  LangGraph'),
    (10.5, '5단계: 보조 조사', 'get_index_summary  |  explain_unassigned\nget_node_logs(RC10-06, …)'),
    ( 8.8, '6단계: 최종 리포트', '모든 tool 결과 통합  →  새 리포트 작성'),
]

for y_pos, label, sub in stages:
    h = 1.1 if '\n' in sub else 0.9
    box(ax, 0.5, y_pos - h, 6.2, h, label, sub,
        color=C_AGENT, fontsize=9)

# 단계 간 화살표
stage_ys = [20.0, 18.5, 17.0, 15.5, 14.0, 10.5, 8.8]
stage_hs = [0.9, 0.9, 0.9, 0.9, 0.9, 1.1, 0.9]
for i in range(len(stage_ys)-1):
    y_top = stage_ys[i] - stage_hs[i]
    y_bot = stage_ys[i+1]
    arrow(ax, 3.6, y_top, 3.6, y_bot, color=C_AGENT, lw=2)

# ════════════════════════════════════════════════════════════
# LANGGRAPH 영역 (오른쪽, x=7.5)
# ════════════════════════════════════════════════════════════
section_bg(ax, 7.3, 6.5, 10.4, 8.0, 'LangGraph  (analyze_logs 내부)', C_LG)

# ── ClickHouse
box(ax, 7.6, 12.8, 3.0, 0.8, 'ClickHouse', 'fetch_logs(time_range)', color=C_CH)

# ── SSH / NodeLogFetcher  ★ 신규
box(ax, 11.2, 12.8, 4.2, 0.8,
    '★ NodeLogFetcher.fetch("_master")',
    'start_dt ~ end_dt  |  severity grep',
    color=C_NEW)

# ── split_by_minute
box(ax, 7.6, 11.5, 3.0, 0.8, 'split_by_minute', '1분 버킷으로 분할', color=C_LG)

# ── analyze_minute 팬아웃
for i, xpos in enumerate([7.6, 9.4, 11.2, 13.0]):
    label = f'analyze_minute' if i < 3 else '…'
    sub   = f'02:0{i}' if i < 3 else ''
    box(ax, xpos, 9.8, 1.7, 0.9, label, sub, color=C_LG, fontsize=8)

# ── synthesize  ★ master_logs 받음
box(ax, 7.6, 7.3, 7.4, 1.1,
    '★ synthesize',
    'findings  +  master_logs  →  시간 연계 분석  →  report',
    color=C_SYNTH)

# ── LangGraph 내부 화살표
# ClickHouse → split
arrow(ax, 9.1, 12.8, 9.1, 12.3, color=C_CH, lw=1.5)
# split → analyze_minute (각각)
for xm in [8.45, 10.25, 12.05, 13.85]:
    arrow(ax, 9.1, 11.5, xm, 10.7, color=C_LG, lw=1.2)
# analyze_minute → synthesize
for xm in [8.45, 10.25, 12.05, 13.85]:
    arrow(ax, xm, 9.8, 9.5+((xm-8.45)*0.3), 8.4, color=C_LG, lw=1.2)
# SSH master_logs → synthesize  ★
arrow(ax, 13.3, 12.8, 11.5, 8.4, color=C_NEW, lw=2,
      label='master_logs')

# ════════════════════════════════════════════════════════════
# analyze_logs tool → LangGraph 연결
# ════════════════════════════════════════════════════════════
# DeepAgent 4단계 → ClickHouse
arrow(ax, 6.7, 13.55, 7.6, 13.2, color=C_CH, lw=1.8, label='fetch_logs')
# DeepAgent 4단계 → SSH master
arrow(ax, 6.7, 13.55, 11.2, 13.2, color=C_NEW, lw=1.8, label='node_info("_master")')

# LangGraph report → DeepAgent 5단계
arrow(ax, 9.3, 7.3, 6.7, 9.7, color=C_SYNTH, lw=1.8, label='report')

# ════════════════════════════════════════════════════════════
# get_node_logs 보조 조사 (오른쪽 하단)
# ════════════════════════════════════════════════════════════
section_bg(ax, 7.3, 0.2, 10.4, 5.8, '보조 조사 (5단계)', C_SSH)

box(ax, 7.6, 4.5, 4.5, 0.9,
    'get_node_logs("RC10-06", …)',
    'start_iso ~ end_iso  |  severity grep',
    color=C_SSH)
box(ax, 12.7, 4.5, 4.8, 0.9,
    'get_node_logs("_master", …)',
    'non-GREEN 일 때  |  더 넓은 구간 가능',
    color=C_SSH)

box(ax, 7.6, 3.0, 9.9, 0.9,
    'get_index_summary  |  explain_unassigned_shards',
    '', color=C_AGENT)

box(ax, 7.6, 1.5, 9.9, 0.9,
    '6단계: 최종 리포트 (재작성)',
    'LangGraph report  +  node logs  +  master logs  →  통합 리포트',
    color=C_SYNTH)

# 보조조사 → 최종리포트
arrow(ax, 9.85, 4.5, 9.85, 3.9, color=C_SSH, lw=1.5)
arrow(ax, 15.1, 4.5, 12.5, 3.9, color=C_SSH, lw=1.5)
arrow(ax, 12.55, 3.0, 12.55, 2.4, color=C_AGENT, lw=1.5)

# 5단계 DeepAgent → 보조조사
arrow(ax, 6.7, 10.0, 7.6, 5.4, color=C_AGENT, lw=1.8)

# ════════════════════════════════════════════════════════════
# 범례
# ════════════════════════════════════════════════════════════
legend_items = [
    (C_AGENT, 'DeepAgent 단계'),
    (C_LG,    'LangGraph 노드'),
    (C_CH,    'ClickHouse'),
    (C_SSH,   'SSH (NodeLogFetcher)'),
    (C_SYNTH, 'synthesize / 리포트'),
    (C_NEW,   '★ 신규 추가 (master_logs)'),
]
for i, (c, label) in enumerate(legend_items):
    x = 0.5 + (i % 3) * 5.8
    y = 0.55 if i < 3 else 0.25
    patch = mpatches.Patch(color=c, label=label)
    ax.add_patch(FancyBboxPatch((x, y), 0.35, 0.22,
                                boxstyle='round,pad=0.02',
                                facecolor=c, edgecolor='none', alpha=0.9, zorder=5))
    ax.text(x + 0.45, y + 0.11, label, fontsize=8, color=C_TEXT,
            va='center', zorder=6)

plt.tight_layout()
plt.savefig('flow_diagram.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('저장 완료: flow_diagram.png')

ModuleNotFoundError: No module named 'matplotlib'